# Assignment 1: Explainable AI for Protein Biology 

**Course:** Advanced Machine Learning for the Life Sciences (MBioAML)
**Dataset:** Extracellular Vesicle (EV) Protein Sorting
**Environment:** Google Colab (CPU runtime is sufficient)

---

## Overview

Machine learning models can predict complex biological outcomes — such as whether a protein is sorted into extracellular vesicles — with high accuracy. But in biology, prediction alone is not enough. We need to understand *why* a model makes a decision: which features drive the prediction, and does that align with known biology?

This assignment puts those ideas into practice. You will:

1. **Train** three models (Logistic Regression, Random Forest, MLP) on real protein biology data
2. **Visualize** global model behavior with PDP, ICE, ALE, and permutation importance
3. **Explain** individual predictions with **LIME** — and explore how its hyperparameters affect results
4. **Explain** with **SHAP** using model-agnostic methods (Sampling, Permutation, Kernel SHAP)
5. **Explain** with **model-specific SHAP** (TreeSHAP, DeepSHAP) and compare speed/reliability
6. **Reflect** on what each method reveals about the biology of EV protein sorting

---

## The Dataset: Extracellular Vesicle Protein Sorting

Extracellular vesicles (EVs) are small membrane-bound particles released by cells. They carry proteins, lipids, and nucleic acids and play key roles in cell-to-cell communication, immune signalling, and even cancer metastasis. Understanding *which proteins* get sorted into EVs — and what makes them EV-destined — is a major open question in cell biology.

**Task:** Binary classification — predict whether a protein will be sorted into EVs (`EV = 1`) or not (`EV = 0`) from protein biochemical and bioinformatic features.

**Features (93 total) include:**
- Physicochemical properties: length, molecular weight, hydrophobicity, isoelectric point, charge, GRAVY index
- Amino acid composition (20 amino acid frequencies)
- Secondary structure fractions (helix, sheet, turn)
- Surface accessibility: total/relative accessible surface area, exposed residue fractions
- Post-translational modifications (PTMs): phosphorylation, ubiquitination, glycosylation, etc.
- Protein domain annotations: coiled-coil, transmembrane, EGF, RRM, etc.
- Solubility and aggregation propensity scores

---

## How to read this notebook

- **Explanatory cells** (like this one) introduce each concept and explain *why* we do things.
- **Code cells** contain the implementation — read them carefully before running.
- Cells marked with ✏️ **STUDENT TASK** require you to write code or answer questions.
- Questions marked **Q** should be answered directly in the notebook (add a markdown cell or use the provided answer box).

> ⚠️ **Colab note:** Run cells top-to-bottom. If your runtime disconnects, restart from Section 0.

---

## Table of Contents

| Section | Topic |
|---------|-------|
| [0. Setup](#setup) | Install packages, set seeds |
| [1. Data](#data) | Load, explore, and understand the EV dataset |
| [2. Models](#models) | Train Logistic Regression, Random Forest, MLP |
| [3. Global Explanations](#global) | PDP, ICE, ALE, Permutation Importance |
| [4. LIME](#lime) | Local explanations, perturbations, kernel, fidelity, complexity |
| [5. SHAP Agnostic](#shap-ag) | Kernel SHAP and background sample effects |
| [6. SHAP Specific](#shap-sp) | TreeSHAP, DeepSHAP; speed & reliability vs Kernel SHAP |
| [7. Synthesis](#synthesis) | Compare methods, biological interpretation |


---
## Section 0: Setup & Installation <a id='setup'></a>

We install all required packages in one go.  It may take 1–2 minutes.

> **Why so many packages?** LIME and SHAP each come with their own dependencies. `torch` is needed for the MLP. `alibi` and `PyALE` provide ALE plots. `interpret` gives us PDP and ICE. `matplotlib` and `seaborn` for all visualization.


In [ ]:
# ── 0.1  Install all required packages ───────────────────────────────────────
# Run this cell first. All output is suppressed for readability.
# If a package fails to install, remove the '> /dev/null 2>&1' and re-run to see the error.

!pip install -q numpy pandas "scikit-learn==1.8.0" matplotlib seaborn > /dev/null 2>&1
!pip install -q torch > /dev/null 2>&1
!pip install -q lime > /dev/null 2>&1
!pip install -q shap > /dev/null 2>&1
!pip install -q PyALE > /dev/null 2>&1
!pip install -q tqdm joblib > /dev/null 2>&1

print("✅ All packages installed.")

In [ ]:
# ── 0.2  Imports and global settings ─────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
import warnings, time, random, copy
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import torch
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch device: {DEVICE}")

# Sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score,
    classification_report, RocCurveDisplay, confusion_matrix
)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.model_selection import train_test_split

# LIME
from lime.lime_tabular import LimeTabularExplainer

# SHAP
import shap

print("✅ All imports successful.")

import ssl
ssl._create_default_https_context = ssl._create_unverified_context

---
## Section 1: Data Loading and Exploration <a id='data'></a>

### 1.1 Loading the data

We train on a large dataset of annotated proteins and evaluate on a held-out test set. The notebook now loads both files directly from this GitHub repository, so no manual upload step is needed when the repository is accessible from Colab.


In [ ]:
# ── 1.1  Load train and test data directly from GitHub ───────────────────
GITHUB_OWNER = "Majeed7"
GITHUB_REPO = "MbioAML_Assignment"
GITHUB_BRANCH = "main"
DATA_SUBDIR = "train%3Atest"  # URL-encoded form of `train:test`

BASE_URL = f"https://raw.githubusercontent.com/{GITHUB_OWNER}/{GITHUB_REPO}/{GITHUB_BRANCH}/{DATA_SUBDIR}"
train_url = f"{BASE_URL}/training_remaining.csv"
test_url = f"{BASE_URL}/test_Dhondt2021.csv"

train_df = pd.read_csv(train_url)
test_df  = pd.read_csv(test_url)

TARGET  = "EV"
ID_COL  = "id"
FEATURE_COLS = [c for c in train_df.columns if c not in [TARGET, ID_COL]]

X_train = train_df[FEATURE_COLS].values.astype(np.float32)
y_train = train_df[TARGET].values
X_test  = test_df[FEATURE_COLS].values.astype(np.float32)
y_test  = test_df[TARGET].values

print(f"Training set : {X_train.shape[0]:,} samples  |  {X_train.shape[1]} features")
print(f"Test set     : {X_test.shape[0]:,} samples")
print(f"Feature list (first 10): {FEATURE_COLS[:10]}")
print(f"Training URL : {train_url}")
print(f"Test URL     : {test_url}")


### 1.2 Class balance

Class imbalance is common in biology datasets. An imbalanced dataset means a model can appear to perform well simply by predicting the majority class. We must check the label distribution before trusting any accuracy number.


In [ ]:
# ── 1.2  Class balance ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (split_name, y) in zip(axes, [("Train", y_train), ("Test (Dhondt2021)", y_test)]):
    counts = np.bincount(y)
    bars = ax.bar(["Non-EV (0)", "EV (1)"], counts, color=["#4C72B0", "#DD8452"], edgecolor="black")
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
                f"{cnt:,}\n({100*cnt/len(y):.1f}%)", ha="center", va="bottom", fontsize=10)
    ax.set_title(f"Label distribution — {split_name}", fontsize=12)
    ax.set_ylabel("Count")

plt.tight_layout()
## if you wanna save the figure, uncomment the line below. Otherwise, it will just be displayed.
# plt.savefig("class_balance.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"Train positive rate: {y_train.mean():.3f}")
print(f"Test  positive rate: {y_test.mean():.3f}")


**Q1.2 -**
a. Is the dataset balanced?
b. How will class imbalance affect model training and evaluation?
c. Which metric is most reliable when classes are imbalanced: accuracy, F1, or AUC?
d. Explain your reasoning.

> *Your answer here*


### Section 1.3: Correlation of features

In [ ]:
# ── 1.4  Correlation heatmap (top 20 features by variance) ──────────────────
top_feat_idx = np.argsort(X_train.var(axis=0))[::-1][:20]
top_feat_names = [FEATURE_COLS[i] for i in top_feat_idx]
corr = pd.DataFrame(X_train[:, top_feat_idx], columns=top_feat_names).corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0,
            annot=False, linewidths=0.3, ax=ax, vmin=-1, vmax=1)
ax.set_title("Pairwise feature correlations (top 20 by variance)", fontsize=13)
plt.tight_layout()
plt.savefig("feature_correlations.png", dpi=120, bbox_inches="tight")
plt.show()

print("Highest absolute correlations:")
corr_long = corr.abs().unstack().sort_values(ascending=False)
corr_long = corr_long[corr_long < 1.0].drop_duplicates()
print(corr_long.head(10).to_string())


---
## Section 2: Model Training <a id='models'></a>

We train three models of increasing complexity:

| Model | Why we use it |
|-------|--------------|
| **Logistic Regression** | giving probabilities; explaining the probabilities of the two classes |
| **Random Forest** | Non-linear ensemble; |
| **MLP (PyTorch)** | Deep network;  |

All models are trained on the same features and evaluated on the same test set. To incorporate class imbalance explicitly, we compute class weights from `y_train` and reuse those weights in Logistic Regression, Random Forest, and the MLP loss.



### 2.1 Logistic Regression
We train a logistic regression in this section. The training has not taken into account the class imbalance (and you need to take care of it!)

In [ ]:
# ── 2.1  Logistic Regression ──────────────────────────────────────────────────
# Compute class weights from the training split only.
class_counts = np.bincount(y_train)
class_weight_dict = {
    0: len(y_train) / (class_counts[0]),
    1: len(y_train) / (class_counts[1]),
}
print(f"Training class counts: Non-EV={class_counts[0]:,}, EV={class_counts[1]:,}")
print(f"Class weights used: {class_weight_dict}")

# We wrap with a StandardScaler — LR is sensitive to feature scale.
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        C=0.4,          # regularization
        random_state=SEED
    ))
])

lr_pipeline.fit(X_train, y_train)
lr_probs = lr_pipeline.predict_proba(X_test)[:, 1]
lr_preds = (lr_probs >= 0.5).astype(int)

print("Logistic Regression — Test set performance")
print(f"  AUC  : {roc_auc_score(y_test, lr_probs):.4f}")
print(f"  F1   : {f1_score(y_test, lr_preds):.4f}")
print(f"  Acc  : {accuracy_score(y_test, lr_preds):.4f}")
print(classification_report(y_test, lr_preds, target_names=["Non-EV","EV"]))


**Q2.1 -**
a. Looking at the performance of Logistic Regression, how does it perform with respect to the Non-EV class (minority)?
b. Can you understand how the classes should be weighted (`class_weight_dict`)?
c. Does adding class weight help the classification? To what extent?

> *Your answer here*


### 2.2 Random Forest


In [ ]:
# ── 2.2  Random Forest ───────────────────────────────────────────────────────
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight="balanced",
    max_features="sqrt",
    n_jobs=-1,
    random_state=SEED
)

rf_model.fit(X_train, y_train)
rf_probs = rf_model.predict_proba(X_test)[:, 1]
rf_preds = (rf_probs >= 0.5).astype(int)

print("Random Forest — Test set performance")
print(f"  AUC  : {roc_auc_score(y_test, rf_probs):.4f}")
print(f"  F1   : {f1_score(y_test, rf_preds):.4f}")
print(f"  Acc  : {accuracy_score(y_test, rf_preds):.4f}")
print(classification_report(y_test, rf_preds, target_names=["Non-EV","EV"]))


**Q2.2 -**
Compare the performance of Random Forest (with and without class weights) with Logistic Regression.

> *Your answer here*


### 2.3 MLP (PyTorch)

We build a small fully-connected network. The architecture is intentionally simple — three hidden layers with ReLU activations and dropout for regularization. We also build a **scikit-learn compatible wrapper** so that LIME (which expects a `predict_proba` function) can use this model without any modifications.


In [ ]:
# ── 2.3a  MLP architecture ────────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

class EVNet(nn.Module):
    """
    Simple 3-layer MLP for EV protein classification.
    Output: single logit (apply sigmoid for probability).
    """
    def __init__(self, input_dim: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            # nn.Linear(64, 32),
            # nn.ReLU(),
            # nn.Linear(32, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)   # shape: (B,)


INPUT_DIM = X_train.shape[1]
mlp = EVNet(INPUT_DIM).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in mlp.parameters() if p.requires_grad)
print(f"MLP: {total_params:,} trainable parameters")
print(mlp)


In [ ]:
# ── 2.3b  Preprocess: scale inputs ────────────────────────────────────────────
# The MLP shares the same scaler as Logistic Regression.
mlp_scaler = StandardScaler()
X_train_scaled = mlp_scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled  = mlp_scaler.transform(X_test).astype(np.float32)

# Use the same training-derived class imbalance information in the MLP loss.
pos_weight = torch.tensor([class_weight_dict[1] / class_weight_dict[0]], dtype=torch.float32).to(DEVICE)
print(f"MLP positive-class loss weight: {pos_weight.item():.3f}")

# Data loaders
train_tensor = TensorDataset(
    torch.tensor(X_train_scaled), torch.tensor(y_train, dtype=torch.float32))
test_tensor  = TensorDataset(
    torch.tensor(X_test_scaled),  torch.tensor(y_test,  dtype=torch.float32))

train_loader = DataLoader(train_tensor, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_tensor,  batch_size=512, shuffle=False)


In [ ]:
# ── 2.3c  Training loop ───────────────────────────────────────────────────────
def train_mlp(model, train_loader, n_epochs=30, lr=1e-3, pos_weight=None):
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

    best_loss = float("inf")
    best_state = None

    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()
        epoch_loss /= len(train_loader)

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_state = copy.deepcopy(model.state_dict())

        if epoch % 10 == 0:
            print(f"  Epoch {epoch:3d}/{n_epochs}  loss={epoch_loss:.4f}")

    model.load_state_dict(best_state)
    return model


print("Training MLP…")
mlp = train_mlp(mlp, train_loader, n_epochs=100, pos_weight=pos_weight)
print("Done.")


In [ ]:
# ── 2.3d  MLP evaluation + sklearn wrapper ────────────────────────────────────
def mlp_predict_proba(X: np.ndarray) -> np.ndarray:
    """
    sklearn-compatible predict_proba for the MLP.
    Accepts a raw (unscaled) numpy array, scales internally.
    Returns shape (N, 2): [P(non-EV), P(EV)].
    """
    X_sc = mlp_scaler.transform(X).astype(np.float32)
    with torch.no_grad():
        logits = mlp(torch.tensor(X_sc).to(DEVICE))
        probs  = torch.sigmoid(logits).cpu().numpy()
    return np.column_stack([1 - probs, probs])


mlp_probs = mlp_predict_proba(X_test)[:, 1]
mlp_preds = (mlp_probs >= 0.5).astype(int)

print("MLP — Test set performance")
print(f"  AUC  : {roc_auc_score(y_test, mlp_probs):.4f}")
print(f"  F1   : {f1_score(y_test, mlp_preds):.4f}")
print(f"  Acc  : {accuracy_score(y_test, mlp_preds):.4f}")
print(classification_report(y_test, mlp_preds, target_names=["Non-EV","EV"]))


### 2.4 Model comparison


In [ ]:
# ── 2.4  Compare all three models ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC curves
for name, probs, color in [
    ("Logistic Regression", lr_probs, "#4C72B0"),
    ("Random Forest",       rf_probs, "#55A868"),
    ("MLP",                 mlp_probs,"#DD8452"),
]:
    RocCurveDisplay.from_predictions(y_test, probs, name=name,
                                      ax=axes[0], color=color)
axes[0].plot([0,1],[0,1],"k--",lw=1)
axes[0].set_title("ROC curves — Test set")

# Bar chart: AUC / F1
models  = ["LR", "RF", "MLP"]
aucs = [roc_auc_score(y_test, p) for p in [lr_probs, rf_probs, mlp_probs]]
f1s  = [f1_score(y_test, p > .5)  for p in [lr_probs, rf_probs, mlp_probs]]

x = np.arange(len(models))
w = 0.35
axes[1].bar(x - w/2, aucs, w, label="AUC",  color="#4C72B0", edgecolor="black")
axes[1].bar(x + w/2, f1s,  w, label="F1",   color="#DD8452", edgecolor="black")
axes[1].set_xticks(x); axes[1].set_xticklabels(models, fontsize=12)
axes[1].set_ylim(0, 1); axes[1].set_ylabel("Score")
axes[1].set_title("AUC and F1 — Test set")
axes[1].legend(); axes[1].axhline(0.5, color="gray", lw=1, ls="--")

plt.tight_layout()
plt.show()


---
## Section 3: Global Explanations — PDP, ICE, ALE, Permutation Importance <a id='global'></a>

In the lectures, we introduced four global explanation methods. Here we apply all four to the EV protein dataset. Each method answers a slightly different question:

| Method | Question answered |
|--------|------------------|
| **Permutation Importance** | Which features, when scrambled, hurt the model most? |
| **PDP** | On average across all proteins, how does feature X affect predicted EV probability? |
| **ICE** | For *individual* proteins, how does the prediction change as we vary feature X? |
| **ALE** | Like PDP, but restricted to realistic feature regions — safer for correlated features |

We use the **Random Forest** as our reference model for this section because it is the best-performing non-linear model and natively supports TreeSHAP later.


### 3.1 Permutation Feature Importance

Permutation importance is model-agnostic and honest: it directly measures how much the model's test performance drops when a feature's information is destroyed. 


In [ ]:
# ── 3.1  Permutation feature importance ──────────────────────────────────────
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    rf_model, X_test, y_test,
    n_repeats=10,
    scoring="roc_auc",
    random_state=SEED,
    n_jobs=-1
)

# Sort by mean importance
perm_mean = perm_result.importances_mean
perm_std  = perm_result.importances_std
order     = np.argsort(perm_mean)[::-1][:20]

fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(
    [FEATURE_COLS[i] for i in order[::-1]],
    perm_mean[order[::-1]],
    xerr=perm_std[order[::-1]],
    color="#4C72B0", edgecolor="black", capsize=4
)
ax.set_xlabel("Mean decrease in AUC when feature is shuffled")
ax.set_title("Permutation Feature Importance — Random Forest (top 20)")
ax.axvline(0, color="gray", lw=1, ls="--")
plt.tight_layout()
## if you wanna save the figure, uncomment the line below. Otherwise, it will just be displayed.
# plt.savefig("permutation_importance.png", dpi=120, bbox_inches="tight")
plt.show()

print("Top 10 features by permutation importance:")
for rank, i in enumerate(order[:10], 1):
    print(f"  {rank:2d}. {FEATURE_COLS[i]:<35s}  {perm_mean[i]:.4f} ± {perm_std[i]:.4f}")


**Q3.1 -**
a. Which features are most important according to permutation importance?
b. Can you interpret 2-3 of the top features biologically?

> *Your answer here*


### 3.2 Partial Dependence Plots (PDP) and ICE

PDPs show the *average* predicted EV probability across the full training set as one feature varies. ICE plots show the same but for *individual proteins*, revealing heterogeneity that the average hides.

Recall the key limitation: PDP averages over the entire data, including potentially unrealistic feature combinations (especially when features are correlated).

---

> ✏️ **STUDENT TASK 3.2:** Implement the PDP and ICE plots for the **top 5 most important features** (by RF feature importance) using scikit-learn's `PartialDependenceDisplay`.
>
> - For the **top row**, produce a standard PDP (`kind="average"`).
> - For the **bottom row**, overlay individual ICE curves with the mean PDP (`kind="both"`).
> - Use a random subsample of 500 training samples as background to keep it fast.
>
> 📖 **Reference:** [sklearn PartialDependenceDisplay documentation](https://scikit-learn.org/stable/modules/partial_dependence.html)
>
> The commented-out code below is a complete solution — try to write it yourself first, then uncomment to check.


In [ ]:
# ── 3.2  Partial Dependence Plots (PDP) and ICE ──────────────────────────────

# Setup (always runs — ALE section below re-uses top_idx)
from sklearn.inspection import PartialDependenceDisplay

N_PDP = 5 # top N_PDP features by RF feature importance for PDP/ICE visualization
importances = rf_model.feature_importances_
top_idx   = np.argsort(importances)[::-1][:N_PDP]
top_names = [FEATURE_COLS[i] for i in top_idx]

rng   = np.random.default_rng(42)
bg_idx = rng.choice(len(X_train), size=min(500, len(X_train)), replace=False)
X_bg  = X_train[bg_idx]

print(f"Top-{N_PDP} features for PDP/ICE: {top_names}")

# ──────────────────────────────────────────────────────────────────────────────
# ✏️ STUDENT TASK 3.2: Plot PDP (top row) and ICE (bottom row) for each feature.
# Use PartialDependenceDisplay.from_estimator with kind="average" for PDP
# and kind="both" for ICE+mean overlay.
# Reference: https://scikit-learn.org/stable/modules/partial_dependence.html
# ──────────────────────────────────────────────────────────────────────────────

# YOUR CODE HERE
# Use a subplot of two rows and N_PDP columns to plot all features together.
fig, axes = plt.subplots(2, N_PDP, figsize=(4 * N_PDP, 8))




**Q3.2 -**
a. Where do the ICE curves spread out compared to the PDP, and what does that spread tell you?
b. For which feature(s) does the PDP look like a flat line but the ICE shows strong individual effects?
c. Do you find PDP reliable for the top 5 features? If yes, for which one(s)? If no, why?
d. Why do we use `X_bg` (a subset of the training set) for the two plots?

> *Your answer here*


### 3.3 Accumulated Local Effects (ALE)

ALE avoids the PDP/ICE problem of unrealistic feature combinations. Instead of varying a feature for *every* protein, it only looks at small intervals and averages the local effect *within* that interval — so it stays close to the real data distribution. For biology, where features are heavily correlated (e.g. many amino acid frequencies sum to 1), ALE is often more trustworthy than PDP.

---

> ✏️ **STUDENT TASK 3.3:** Implement ALE plots for the same top 5 features using the `PyALE` package.
>
> - Call `PyALE.ale(...)` with `plot=False` to get the ALE result as a DataFrame, then plot it manually.
> - Include 95% confidence intervals (`include_CI=True, C=0.95`).
> - Use the training data (as a DataFrame with column names) as input to `PyALE.ale`.

> - Reference: https://pypi.org/project/PyALE/


In [ ]:
# ── 3.3  Accumulated Local Effects (ALE) ─────────────────────────────────────
# top_idx and N_PDP are already defined in cell 3.2 above.

# ✏️ STUDENT TASK 3.3: Write your ALE code here using PyALE.
# Use the top features in top_idx / top_names from the cell above.
# Reference: https://pypi.org/project/PyALE/

# YOUR CODE HERE

import PyALE


**Q3.3 -**
a. Compare the PDP and ALE plots for the same features.
b. Do they tell the same story?
c. If they differ, which features are most correlated with the feature being plotted, and how could that correlation make PDP misleading?

> *Your answer here*


---
## Section 4: LIME — Local Interpretable Model-Agnostic Explanations <a id='lime'></a>

LIME explains **one prediction at a time**. For a protein you want to understand, LIME:
1. Creates many slightly perturbed copies of that protein (changing feature values)
2. Asks the model for predictions on all those perturbed copies
3. Weights perturbed copies by their distance to the original protein (closer = higher weight)
4. Fits a simple **sparse linear model** on the weighted perturbed dataset
5. The coefficients of that linear model are the feature attributions

LIME is **model-agnostic** — it treats any model as a black box and only calls `predict_proba`.

We will use the **Random Forest** as our primary model in this section (also try the MLP if you want).

---

### 4.1 Setting up the LIME explainer and helper functions

The `LimeTabularExplainer` needs:
- Training data (to understand feature distributions and ranges)
- Feature names and mode (`classification` or `regression`)
- Class names (for display)

Check `make_lime_explainer` helper function

In [ ]:
# Use one fixed split for LIME analysis to keep experiments comparable
X_lime = X_test
y_lime = y_test
rf_probs_lime = rf_model.predict_proba(X_lime)[:, 1]

# Select representative samples for analysis
ev_mask = y_lime == 1
nonev_mask = y_lime == 0

tp_candidates = np.where((rf_probs_lime >= 0.7) & ev_mask)[0]
fn_candidates = np.where((rf_probs_lime < 0.4) & ev_mask)[0]
tn_candidates = np.where((rf_probs_lime < 0.3) & nonev_mask)[0]

tp_idx = int(tp_candidates[0]) if len(tp_candidates) else int(np.where(ev_mask)[0][np.argmax(rf_probs_lime[ev_mask])])
fn_idx = int(fn_candidates[0]) if len(fn_candidates) else int(np.where(ev_mask)[0][np.argmin(rf_probs_lime[ev_mask])])
tn_idx = int(tn_candidates[0]) if len(tn_candidates) else int(np.where(nonev_mask)[0][np.argmin(rf_probs_lime[nonev_mask])])

sample_indices = {"TP": tp_idx, "FN": fn_idx, "TN": tn_idx}
for tag, idx in sample_indices.items():
    print(f"{tag} sample idx={idx:4d} | y={y_lime[idx]} | P(EV)={rf_probs_lime[idx]:.3f}")


#### LIME explainer factory ####

def make_lime_explainer(kernel_width=None, seed=42):
    """Create a LimeTabularExplainer with the given kernel_width and random seed."""
    return LimeTabularExplainer(
        training_data=X_train,
        feature_names=FEATURE_COLS,
        class_names=["non-EV", "EV"],
        mode="classification",
        kernel_width=kernel_width,
        discretize_continuous=True,
        random_state=seed,
    )

# NOTE: We work directly with exp.as_list(label=1) which returns
# (feature_description_string, weight) pairs. Each string describes
# a discretized bin, e.g. "length <= 250.00" or "gravy > 0.30".
# The weight is the linear coefficient of that bin in the local surrogate model.
# Positive weight  → this bin pushes the prediction toward EV (class 1).
# Negative weight  → this bin pushes the prediction away from EV.
print("LIME helpers ready.")


### 4.2 Let's visualize the explanation for one instance

We first visualize the explanation of an instance.

> ✏️ **STUDENT TASK 4.2:** The explanation is derived; can you plot the explanation for the instance? 

Also print the most important features for the instance under explanation.


In [ ]:
# ── 4.2  Feature attribution visualization (single sample) ───────────────────
NUM_FEATURES_LIME = 15
NUM_SAMPLES_LIME  = 500
DEFAULT_KW = np.sqrt(len(FEATURE_COLS)) * 0.75

explainer_default = make_lime_explainer(kernel_width=DEFAULT_KW, seed=42)
exp_tp = explainer_default.explain_instance(
    data_row    = X_lime[tp_idx],
    predict_fn  = rf_model.predict_proba,
    num_features= NUM_FEATURES_LIME,
    num_samples = NUM_SAMPLES_LIME,
)

# YOUR CODE HERE


**Q4.2 -**
Which features are most important for this specific instance?

> *Your answer here*


### 4.3 LIME Instability

**Why is LIME unstable?**

Every time LIME runs it generates a *fresh* random neighborhood around the instance. Because the neighborhood is sampled randomly, two runs with different random seeds will produce slightly different sets of perturbed points — and therefore slightly different local linear models. This means the feature attributions can change between runs even for the *same instance and the same model*.

The key parameters that control how much this randomness matters are:

- **`num_samples`** — the number of perturbed copies LIME creates. More samples → the local linear fit is estimated from more data → less run-to-run variance. Think of it like increasing sample size in a statistical estimate: with only 50 perturbations, each random draw has a big impact on the fit; with 500, individual draws average out.
- **`random_state`** — setting a fixed seed removes the randomness entirely, but gives a false sense of stability: you get the same answer every run, but it may not be the *right* answer.

**Rule of thumb:** `num_samples = 500` is a reasonable default — fast enough for interactive use and stable enough for most explanations. Below we will verify this claim.

---

> ✏️ **STUDENT TASK 4.3:** Run the stability experiment below with **30 different random seeds**. 
> The code is ready to run — just execute the cell and interpret the output.
> After running, answer the questions in the markdown cell that follows.


In [ ]:
# ── 4.3  LIME stability across random seeds ───────────────────────────────────
N_RUNS = 30          # number of independent runs (different random seeds)
TOP_N  = 10          # compare top-10 features across runs
NUM_SAMPLES_STABILITY = 500   # num_samples per run

run_weights = {}   # feature_description -> list of weights across runs

for run in range(N_RUNS):
    exp_run = LimeTabularExplainer(
        training_data         = X_train,
        feature_names         = FEATURE_COLS,
        class_names           = ["Non-EV", "EV"],
        mode                  = "classification",
        discretize_continuous = True,
        random_state          = run   # different seed each run
    ).explain_instance(
        data_row     = X_test[tp_idx],
        predict_fn   = rf_model.predict_proba,
        num_features = TOP_N,
        num_samples  = NUM_SAMPLES_STABILITY,
    )
    for feat_str, weight in exp_run.as_list(label=1):
        run_weights.setdefault(feat_str, []).append(weight)

# Build summary: features that appear in at least half the runs
stable_feats = {k: v for k, v in run_weights.items() if len(v) >= N_RUNS // 2}
feat_means   = {k: np.mean(v) for k, v in stable_feats.items()}
feat_stds    = {k: np.std(v)  for k, v in stable_feats.items()}
sorted_feats = sorted(feat_means.items(), key=lambda x: abs(x[1]), reverse=True)[:15]

fig, ax = plt.subplots(figsize=(11, 5))
names  = [f[0][:45] for f in sorted_feats]
means  = [f[1] for f in sorted_feats]
stds   = [feat_stds.get(f[0], 0) for f in sorted_feats]
x_pos  = range(len(names))

bars = ax.bar(x_pos, means, yerr=stds, capsize=4,
              color=["#DD8452" if m > 0 else "#4C72B0" for m in means],
              edgecolor="black", alpha=0.85)
ax.set_xticks(x_pos)
ax.set_xticklabels(names, rotation=45, ha="right", fontsize=8)
ax.set_ylabel(f"Mean LIME weight ± std across {N_RUNS} runs")
ax.set_title(f"LIME stability: protein {tp_idx} — feature weights over {N_RUNS} runs (num_samples={NUM_SAMPLES_STABILITY})")
ax.axhline(0, color="gray", lw=1, ls="--")
plt.tight_layout()
# plt.savefig("lime_stability.png", dpi=120, bbox_inches="tight")
plt.show()

# Rank overlap between run 0 and run N-1
run0_feats  = [f for f, _ in sorted(run_weights.items(),
               key=lambda x: abs(x[1][0]) if x[1] else 0, reverse=True)[:TOP_N]]
runN_feats  = [f for f, _ in sorted(run_weights.items(),
               key=lambda x: abs(x[1][-1]) if len(x[1]) > 1 else 0, reverse=True)[:TOP_N]]
overlap = len(set(run0_feats) & set(runN_feats))
print(f"Top-{TOP_N} feature overlap between run 0 and run {N_RUNS-1}: {overlap}/{TOP_N}")


**Q4.3a -**
a. Looking at the error bars, which features are stable across runs (small error bar)?
b. Which features are unstable across runs (large error bar)?

> *Your answer here*

---

> ✏️ **STUDENT TASK 4.8:** Run the stability analysis for the MLP instead of the Random Forest. Does the MLP give more or less stable LIME explanations? Why might that be?


### 4.4 Effect of number of perturbations (`num_samples`)

The number of perturbations controls how well LIME estimates the local decision boundary. 

**What `num_samples` does:**  
LIME fits a weighted linear model on `num_samples` randomly perturbed copies of the input. The more copies, the more information the linear model has → the fit is more stable. The flip side: each call to `predict_proba` takes time, so large `num_samples` slows things down.

- Too few samples (e.g. 50) → noisy linear fit → feature weights jump between runs
- Too many samples (e.g. 10 000) → very stable, but slow — likely overkill
- **`num_samples = 500`** is a good practical default: stable and fast

The plot below shows how the weight assigned to each feature changes as we increase `num_samples` from 50 to 1 000. Features whose lines flatten out quickly are those that are robustly identified even with few perturbations.

> ✏️ **STUDENT TASK 4.4:** Run the cell below and inspect the convergence plot.  
> At what value of `num_samples` do the feature weights appear to stabilise?  
> Answer the questions in the markdown cell that follows.


In [ ]:
# ── 4.3  Effect of num_samples on LIME explanation ───────────────────────────
# We vary num_samples from 50 to 1000 and track how feature weights change.
# The reference value we recommend for the rest of this notebook is 500.

sample_sizes = [50, 100, 200, 300, 500, 1000]
TOP_FEAT_NAME = FEATURE_COLS[order[0]]   # most important feature from permutation importance

lime_weights_by_n = {}

for n in sample_sizes:
    exp = explainer_default.explain_instance(
        data_row     = X_test[tp_idx],
        predict_fn   = rf_model.predict_proba,
        num_features = len(FEATURE_COLS),
        num_samples  = n,
    )
    lime_weights_by_n[n] = dict(exp.as_list(label=1))

# Track the weight of up to 8 of the top features across sample sizes
feat_count = 8
TRACK_FEATS_PARTIAL = [f for f in lime_weights_by_n[1000].keys()
                       if any(tf in f for tf in [FEATURE_COLS[order[i]] for i in range(feat_count)])][:feat_count]
if not TRACK_FEATS_PARTIAL:
    TRACK_FEATS_PARTIAL = list(lime_weights_by_n[1000].keys())[:feat_count]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860", "#DA8BC3", "#8C8C8C"]
for feat, color in zip(TRACK_FEATS_PARTIAL, colors):
    vals = [lime_weights_by_n[n].get(feat, np.nan) for n in sample_sizes]
    ax.plot(sample_sizes, vals, "o-", color=color, lw=2, label=feat[:40])

ax.axvline(500, color="gray", lw=1.5, ls="--", label="recommended: 500")
ax.set_xscale("log")
ax.set_xlabel("Number of perturbations (num_samples)", fontsize=11)
ax.set_ylabel("LIME feature weight", fontsize=11)
ax.set_title("LIME stability: feature weight vs number of perturbations", fontsize=12)
ax.legend(fontsize=8, loc="upper right")
ax.axhline(0, color="black", lw=0.8, ls=":")
plt.tight_layout()
plt.show()


**Q4.4 -**
a. How can different numbers of perturbations affect the feature importance in LIME? Can this be resolved?
b. If you repeat executing this cell, do you see the same plot?

> *Your answer here*


### 4.5 Effect of kernel width (distance / neighborhood size)

LIME uses a **kernel function** to weight perturbations by their distance to the original sample. Closer perturbations get higher weight. The **kernel width** controls the *size* of the local neighborhood:

- **Small kernel width** → only very nearby perturbations count → very local, but noisy
- **Large kernel width** → distant perturbations also count → smoother, but loses locality

In `lime`, the kernel width can be set via the `kernel_width` parameter of `LimeTabularExplainer`. The default is `np.sqrt(n_features) * 0.75`.

> ✏️ **STUDENT TASK 4.5:** Use the implementation of the previous sections and implement LIME explainer with different kernel width. 


In [ ]:
# ── 4.4  Effect of kernel_width on LIME explanation ──────────────────────────

# Your code goes here


**Q4.5 -**
a. How do the feature attributions change as you increase the kernel width?
b. At very small kernel width, why might the explanation be noisy?
c. At very large kernel width, the local linear model effectively approximates the global behavior. Is that still useful?
d. How does this relate to the LIME limitation discussed in the lecture: "There is no principled rule for choosing this width"?

> *Your answer here*


---

## Section 5: SHAP 

In this section, we first compute Shapley values by hand for a simple 3-feature game. SHAP applies the same idea to real machine learning models: each feature gets a Shapley value that measures how much it *contributes* to pushing the prediction away from the baseline.

In this section you will:
1. **Study** how Kernel SHAP is sensitive to the background set and to `nsamples` (analogous to LIME's `num_samples`)
2. **Compare** Kernel SHAP with the exact TreeSHAP for the Random Forest
3. **Apply** DeepSHAP to the MLP and compare the feature rankings

We keep the three trained models exactly as they are. The focus is on three SHAP settings:
- **Kernel SHAP** on the **Random Forest** — model-agnostic reference
- **TreeSHAP** on the **Random Forest** — exact, tree-specific method
- **DeepSHAP** on the **MLP** — neural-network-specific approximation

> ⚠️ **Important:** Run the helper cell (5.0) first — it sets up all shared functions and selects a reference protein for the local explanation experiments.

---

> ✏️ **HOMEWORK OVERVIEW**
> - Section 5.1: Compute Shapley value for an examplary game
> - Section 5.1: Understand one local explanation via a force plot
> - Section 5.2: Investigate how background set size affects Kernel SHAP
> - Section 5.3: Investigate how `nsamples` affects Kernel SHAP
> - Section 5.4: Run Kernel SHAP globally and summarize feature importance
> - Section 6.1: Run TreeSHAP and compare to Kernel SHAP
> - Section 6.2: Run DeepSHAP on the MLP




In [ ]:

# ── 5.0  SHAP helpers and reference protein ───────────────────────────────────
from scipy.stats import spearmanr
from IPython.display import display

shap.initjs()


def select_shap_output(values, class_idx=1):
    """Handle old/new SHAP return formats and extract one output."""
    if isinstance(values, list):
        return np.asarray(values[class_idx], dtype=float)

    arr = np.asarray(values, dtype=float)

    if arr.ndim == 3:
        if arr.shape[-1] > class_idx and arr.shape[-1] <= 10:
            return arr[..., class_idx]
        if arr.shape[0] > class_idx and arr.shape[0] <= 10:
            return arr[class_idx]

    return arr


def select_base_value(base_values, class_idx=1):
    """Extract a scalar expected value from SHAP explainers."""
    arr = np.asarray(base_values, dtype=float)

    if arr.ndim == 0:
        return float(arr)
    if arr.ndim == 1:
        if arr.size > class_idx:
            return float(arr[class_idx])
        return float(arr[0])

    flat = arr.reshape(-1)
    return float(flat[class_idx if flat.size > class_idx else 0])


def make_explanation(values, base_values, data, feature_names):
    """Create a SHAP Explanation object with one scalar base value per sample."""
    values = np.asarray(values, dtype=float)
    data = np.asarray(data, dtype=float)

    if values.ndim == 1:
        values = values.reshape(1, -1)
    if data.ndim == 1:
        data = data.reshape(1, -1)

    base_scalar = float(base_values)
    base_vector = np.full(values.shape[0], base_scalar, dtype=float)

    return shap.Explanation(
        values=values,
        base_values=base_vector,
        data=data,
        feature_names=feature_names,
    )


def top_contributors(explanation, row=0, top_k=10):
    """Tabular view of the strongest positive/negative SHAP contributions."""
    df = pd.DataFrame({
        "feature": explanation.feature_names,
        "feature_value": explanation.data[row],
        "shap_value": explanation.values[row],
    })
    df["abs_shap"] = df["shap_value"].abs()
    return df.sort_values("abs_shap", ascending=False).head(top_k).drop(columns="abs_shap")


def plot_mean_abs_shap(mean_abs_values, title, color, top_k=15):
    order = np.argsort(mean_abs_values)[::-1][:top_k]
    plt.figure(figsize=(8, 6))
    plt.barh(
        [FEATURE_COLS[i][:28] for i in order[::-1]],
        mean_abs_values[order[::-1]],
        color=color,
        edgecolor="black",
        alpha=0.9,
    )
    plt.xlabel("Mean |SHAP value|")
    plt.title(title)
    plt.tight_layout()
    plt.show()


def rf_predict_ev(X):
    X = np.asarray(X, dtype=np.float32)
    return rf_model.predict_proba(X)[:, 1]


rf_probs_test = rf_predict_ev(X_test)
ev_test_idx = np.where(y_test == 1)[0]

if len(ev_test_idx):
    ref_idx = int(ev_test_idx[np.argmax(rf_probs_test[ev_test_idx])])
else:
    ref_idx = int(np.argmax(rf_probs_test))

reference_instance = X_test[ref_idx:ref_idx + 1]
print(f"Reference protein index : {ref_idx}")
print(f"True label              : {y_test[ref_idx]}")
print(f"Random Forest P(EV)     : {rf_probs_test[ref_idx]:.4f}")

# Kernel SHAP is expensive in the full 93-dimensional space.
# For the sensitivity experiments below, we restrict to the 12 most important RF features
# and keep the remaining features fixed at the reference protein values.
kernel_top_idx = np.argsort(rf_model.feature_importances_)[::-1][:12]
kernel_top_names = [FEATURE_COLS[i] for i in kernel_top_idx]
reference_top = reference_instance[:, kernel_top_idx]


def rf_predict_ev_top(X_sub):
    X_sub = np.asarray(X_sub, dtype=np.float32)
    full = np.repeat(reference_instance, repeats=X_sub.shape[0], axis=0)
    full[:, kernel_top_idx] = X_sub
    return rf_predict_ev(full)


# Shared subset for global SHAP comparisons on the Random Forest and MLP.
RNG = np.random.default_rng(SEED)
N_EXPLAIN_RF = min(40, len(X_test))
rf_explain_idx = RNG.choice(len(X_test), size=N_EXPLAIN_RF, replace=False)
X_explain_rf = X_test[rf_explain_idx]



### Section 5.1: Shapley value for a cooperative game
Before we move to SHAP (which computes Shapley values for machine learning models), let's make sure we understand what a **Shapley value** is and how it is calculated — through a small example used in the lecture.


---

> ✏️ **STUDENT TASK:** Implement the function `compute_shapley_values` below that takes a list of players and a characteristic function dictionary, and returns the Shapley value for each player.
>
> Then apply it to the example above and verify that the values sum to 10.
>
> A complete solution is provided in the comments — try writing it yourself first!


In [ ]:
from itertools import permutations
from math import factorial

# ── Lecture example: characteristic function ─────────────────────────────────
players = ["f1", "f2", "f3"]

characteristic_function = {
    frozenset():             0,
    frozenset(["f1"]):       1,
    frozenset(["f2"]):       1,
    frozenset(["f3"]):       2,
    frozenset(["f1","f2"]):  4,
    frozenset(["f1","f3"]):  5,
    frozenset(["f2","f3"]):  6,
    frozenset(["f1","f2","f3"]): 10,
}

# ── ✏️ STUDENT TASK ───────────────────────────────────────────────────────────
# Implement compute_shapley_values below.
# Then call it with the players and characteristic_function above
# and verify that the values sum to 10.

def compute_shapley_values(players, char_func):
    """
    Compute the Shapley value for each player.

    Parameters
    ----------
    players   : list of player names
    char_func : dict mapping frozenset -> coalition value

    Returns
    -------
    dict mapping player -> Shapley value
    """
    # YOUR CODE HERE
    pass


# Call your function and print the results:
# phi = compute_shapley_values(players, characteristic_function)
# for p, v in phi.items():
#     print(f"  phi({p}) = {v:.4f}")
# print(f"  Sum = {sum(phi.values()):.4f}  (should be {characteristic_function[frozenset(players)] - characteristic_function[frozenset()]})")





### 5.1 Start with one local explanation: a force plot

Before looking at summary plots, it helps to see how SHAP explains **one prediction**.

A **force plot** answers:
> Starting from the model's **baseline prediction**, which features push this protein **toward EV** (positive SHAP values), and which push it **away from EV** (negative SHAP values)?

We use **TreeSHAP** here because it is fast and exact for tree models.

> ✏️ **STUDENT TASK 5.1:** Make a TreeExplainer and use the force plot in SHAP and plot the result of the explanation.


In [ ]:

# ── 5.1  Force plot for one protein (TreeSHAP) ───────────────────────────────

# Your code goes here


**Q5.1 -**
a. Which features have the largest positive SHAP values for this protein?
b. Which features have the largest negative SHAP values?
c. Does the baseline plus the signed feature contributions recover the final prediction, and why is that additive decomposition useful?

> *Your answer here*


### 5.2 Kernel SHAP — sensitivity to the background set

Kernel SHAP is **model-agnostic**: it only needs a prediction function. The price is that it must simulate feature absence by integrating over a **background dataset**. That background is a real modeling choice:

- A **small** background is fast, but may be noisy or unrepresentative.
- A **larger** background gives a better approximation of the reference distribution, but is slower.

To keep the experiment feasible on CPU, we explain the same reference protein using only the **top 12 Random Forest features** (the rest are held fixed at the protein's own values).

> ✏️ **STUDENT TASK 5.2:** Run the cell below to see how the SHAP values change as the background size increases from 10 to 200. Then answer the questions that follow.


In [ ]:

# ── 5.2  Kernel SHAP: effect of background size ─────────────────────────────
print("Kernel SHAP background-set experiment")

background_sizes = [10, 25, 50, 100, 200]
kernel_bg_values = {}
kernel_bg_runtime = {}
fixed_nsamples = 200

for bg_size in background_sizes:
    rng_bg = np.random.default_rng(SEED + bg_size)
    bg_idx = rng_bg.choice(len(X_train), size=bg_size, replace=False)
    background_sub = X_train[bg_idx][:, kernel_top_idx]

    t0 = time.time()
    explainer_bg = shap.KernelExplainer(rf_predict_ev_top, background_sub, link="identity")
    phi_bg = np.asarray(explainer_bg.shap_values(reference_top, nsamples=fixed_nsamples), dtype=float).reshape(-1)
    kernel_bg_runtime[bg_size] = time.time() - t0
    kernel_bg_values[bg_size] = phi_bg
    print(f"  background={bg_size:3d} | runtime={kernel_bg_runtime[bg_size]:6.2f}s")

bg_heatmap = pd.DataFrame(kernel_bg_values, index=kernel_top_names).T
bg_summary = pd.DataFrame({
    "background_size": background_sizes,
    "runtime_s": [kernel_bg_runtime[s] for s in background_sizes],
})

display(bg_summary.style.format({"runtime_s": "{:.2f}"}))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.heatmap(
    bg_heatmap,
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".3f",
    ax=axes[0],
)
axes[0].set_title("Kernel SHAP values for one protein, changing only the background size")
axes[0].set_xlabel("Top RF features")
axes[0].set_ylabel("Background size")

axes[1].plot(bg_summary["background_size"], bg_summary["runtime_s"], marker="o", lw=2, color="#DD8452")
axes[1].set_xlabel("Background size")
axes[1].set_ylabel("Runtime (s)")
axes[1].set_title("Runtime as background size increases")
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()


**Q5.2 -**
a. Which features change the most when the background size changes?
b. Why does an unrepresentative background set change the SHAP values?
c. Looking at both the heatmap and runtime plot, what looks like a reasonable background size for this dataset?

> *Your answer here*


### 5.3 Kernel SHAP — sensitivity to `nsamples`

Kernel SHAP also depends on how many coalitions it samples, controlled by `nsamples`.

This is **directly analogous to LIME's `num_samples`**: both control the number of random perturbations used to estimate the local linear approximation. Larger `nsamples` → better approximation → longer runtime.

- At low `nsamples` the SHAP values can be noisy because the weighted regression is under-determined
- At high `nsamples` the values stabilise, but computation time grows linearly

Here we fix the background set (100 random training proteins) and only change `nsamples`.

> ✏️ **STUDENT TASK 5.3:** Run the cell below and inspect the heatmap and runtime-vs-accuracy plot. Then answer the questions that follow.


In [ ]:

# ── 5.3  Kernel SHAP: effect of nsamples ────────────────────────────────────
print("Kernel SHAP nsamples experiment")

sample_budgets = [50, 100, 200, 400, 800]
fixed_bg_idx = np.random.default_rng(SEED).choice(len(X_train), size=100, replace=False)
fixed_background_sub = X_train[fixed_bg_idx][:, kernel_top_idx]

kernel_ns_values = {}
kernel_ns_runtime = {}
explainer_ns = shap.KernelExplainer(rf_predict_ev_top, fixed_background_sub, link="identity")

for ns in sample_budgets:
    t0 = time.time()
    phi_ns = np.asarray(explainer_ns.shap_values(reference_top, nsamples=ns), dtype=float).reshape(-1)
    kernel_ns_runtime[ns] = time.time() - t0
    kernel_ns_values[ns] = phi_ns
    print(f"  nsamples={ns:4d} | runtime={kernel_ns_runtime[ns]:6.2f}s")

ref_ns = sample_budgets[-1]
ns_heatmap = pd.DataFrame(kernel_ns_values, index=kernel_top_names).T

ns_summary = pd.DataFrame({
    "nsamples": sample_budgets,
    "runtime_s": [kernel_ns_runtime[s] for s in sample_budgets],
    "spearman_vs_largest_nsamples": [
        spearmanr(np.abs(kernel_ns_values[s]), np.abs(kernel_ns_values[ref_ns])).statistic
        for s in sample_budgets
    ],
    "mae_vs_largest_nsamples": [
        np.mean(np.abs(kernel_ns_values[s] - kernel_ns_values[ref_ns]))
        for s in sample_budgets
    ],
})

display(
    ns_summary.style.format({
        "runtime_s": "{:.2f}",
        "spearman_vs_largest_nsamples": "{:.3f}",
        "mae_vs_largest_nsamples": "{:.4f}",
    })
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.heatmap(
    ns_heatmap,
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".3f",
    ax=axes[0],
)
axes[0].set_title("Kernel SHAP values for one protein\nchanging only nsamples")
axes[0].set_xlabel("Top RF features")
axes[0].set_ylabel("nsamples")

axes[1].plot(ns_summary["nsamples"], ns_summary["runtime_s"], marker="o", lw=2, color="#DD8452", label="Runtime")
axes[1].plot(ns_summary["nsamples"], ns_summary["mae_vs_largest_nsamples"], marker="s", lw=2, color="#4C72B0", label="MAE vs largest nsamples")
axes[1].set_xlabel("nsamples")
axes[1].set_title("Approximation quality vs runtime")
axes[1].legend()

plt.tight_layout()
plt.show()


**Q5.3 -**
a. At low `nsamples`, which feature attributions look unstable?
b. At what point do the SHAP values appear to stabilize for this protein?
c. If you had to explain 1,000 proteins on CPU, how would you choose `nsamples` in practice?

> *Your answer here*


### 5.4 Kernel SHAP on a subset of proteins

Now that we have calibrated the background size and `nsamples`, we run Kernel SHAP on a small subset of test proteins and summarize the results with a **global importance bar chart**.

> ✏️ **STUDENT TASK 5.4:** Run the cell below to compute Kernel SHAP globally on 40 test proteins.  
> Then use the provided code snippet in the following markdown cell to produce a **beeswarm plot** and answer the questions.


In [ ]:

# ── 5.4  Kernel SHAP on a subset of test proteins ───────────────────────────
KERNEL_BACKGROUND_SIZE = min(100, len(X_train))
KERNEL_NSAMPLES_GLOBAL = 200

bg_idx_100 = np.random.default_rng(SEED + 100).choice(len(X_train), size=KERNEL_BACKGROUND_SIZE, replace=False)
background_100 = X_train[bg_idx_100]

print("Computing Kernel SHAP on a subset of test proteins...")
t0 = time.time()

kernel_explainer_rf = shap.KernelExplainer(rf_predict_ev, background_100, link="identity")
kernel_raw_rf = kernel_explainer_rf.shap_values(X_explain_rf, nsamples=KERNEL_NSAMPLES_GLOBAL)
kernel_values_rf = np.asarray(kernel_raw_rf, dtype=float)
if kernel_values_rf.ndim == 1:
    kernel_values_rf = kernel_values_rf.reshape(1, -1)

kernel_base_rf = select_base_value(kernel_explainer_rf.expected_value, class_idx=0)
shap_exp_kernel_rf = make_explanation(
    values=kernel_values_rf,
    base_values=kernel_base_rf,
    data=X_explain_rf,
    feature_names=FEATURE_COLS,
)

mean_kernel = np.abs(shap_exp_kernel_rf.values).mean(axis=0)
t_kernel = time.time() - t0

print(f"Kernel SHAP runtime: {t_kernel:.2f}s for {len(X_explain_rf)} proteins")
plot_mean_abs_shap(mean_kernel, "Kernel SHAP on Random Forest (global summary)", "#4C72B0", top_k=15)

display(
    pd.DataFrame({
        "feature": FEATURE_COLS,
        "mean_abs_shap": mean_kernel,
    })
    .sort_values("mean_abs_shap", ascending=False)
    .head(12)
    .style.format({"mean_abs_shap": "{:.4f}"})
)



> ✏️ **Student activity: Kernel SHAP beeswarm**
>
> Use the explanation object from the previous cell and compare a beeswarm to the bar plot:
>
> ```python
> plt.figure(figsize=(10, 7))
> shap.plots.beeswarm(shap_exp_kernel_rf, max_display=20, show=False)
> plt.title("Kernel SHAP beeswarm — Random Forest")
> plt.tight_layout()
> plt.show()
> ```
>


In [ ]:
# Your code goes here

**Q5.4 -**
a. Which features have large average impact?
b. Which features show a wide spread of effects across proteins?
c. For the top feature, do high feature values usually push predictions toward EV or away from EV?

> *Your answer here*


---
## Section 6: Model-specific SHAP <a id='shap-sp'></a>

Model-specific SHAP methods use the internal structure of the model instead of treating it as a black box. They are usually better defaults when available:
- **TreeSHAP** — exact and fast for tree ensembles (no sampling approximation needed)
- **DeepSHAP** — an approximation tailored to neural networks, using backpropagation-style attribution

> ✏️ **STUDENT TASK 6.1:** Implement TreeSHAP for explaining the random forest trained above.

In [ ]:
## Your code goes here

# just have a variable like:
# mean_tree_all = np.abs(shap_exp_tree.values).mean(axis=0)
# we need it for comparison to other methods
mean_tree_all = np.array([]) # placeholder to avoid errors in later cells; replace with actual values from your TreeSHAP explanation

**Q6.1 -**
a. How much faster is TreeSHAP than Kernel SHAP on the same proteins?
b. Which top features are shared by both methods?
c. Which features disagree the most, and what might explain that disagreement?
d. For a Random Forest in a real biology paper, which explainer would you choose by default?

> *Your answer here*


### 6.2 DeepSHAP for the MLP

DeepSHAP uses the network's structure (specifically, backpropagation-like rules) to attribute contributions through each layer. Like Kernel SHAP, it still needs a **background set** to define the baseline expectation. Unlike TreeSHAP, it is still an **approximation** — the network's non-linearities and interactions are not handled exactly.

> ✏️ **STUDENT TASK 6.2:** Run the DeepSHAP cell below, then answer the questions that follow.


In [ ]:

# ── 6.2  DeepSHAP for the MLP ───────────────────────────────────────────────
print("Computing DeepSHAP on the MLP...")
t0_deep = time.time()

class MLPProbabilityWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        logits = self.model(x)
        return torch.sigmoid(logits).unsqueeze(-1)


mlp_wrapper = MLPProbabilityWrapper(mlp).to(DEVICE)
mlp_wrapper.eval()

DEEP_BACKGROUND_SIZE = min(100, len(X_train_scaled))
bg_idx_deep = np.random.default_rng(SEED + 200).choice(len(X_train_scaled), size=DEEP_BACKGROUND_SIZE, replace=False)
bg_deep = torch.tensor(X_train_scaled[bg_idx_deep], dtype=torch.float32).to(DEVICE)
X_explain_deep = torch.tensor(X_test_scaled[rf_explain_idx], dtype=torch.float32).to(DEVICE)

deep_explainer = shap.DeepExplainer(mlp_wrapper, bg_deep)
deep_raw = deep_explainer.shap_values(X_explain_deep, check_additivity=False)

if isinstance(deep_raw, list):
    deep_values = np.asarray(deep_raw[0], dtype=float)
else:
    deep_values = np.asarray(deep_raw, dtype=float)

if deep_values.ndim == 3 and deep_values.shape[-1] == 1:
    deep_values = deep_values[..., 0]
if deep_values.ndim == 1:
    deep_values = deep_values.reshape(1, -1)

shap_deep_np = deep_values
mean_deep = np.abs(shap_deep_np).mean(axis=0)
deep_base = select_base_value(deep_explainer.expected_value, class_idx=0)

shap_exp_deep = make_explanation(
    values=shap_deep_np,
    base_values=deep_base,
    data=X_test[rf_explain_idx],
    feature_names=FEATURE_COLS,
)

t_deep = time.time() - t0_deep
print(f"DeepSHAP runtime: {t_deep:.2f}s for {len(rf_explain_idx)} proteins")
plot_mean_abs_shap(mean_deep, "DeepSHAP on the MLP (global summary)", "#C44E52", top_k=15)

display(
    pd.DataFrame({
        "feature": FEATURE_COLS,
        "mean_abs_shap": mean_deep,
    })
    .sort_values("mean_abs_shap", ascending=False)
    .head(12)
    .style.format({"mean_abs_shap": "{:.4f}"})
)


**Q6.2 -**
a. Which features does the MLP rely on most strongly?
b. Do those features overlap with the Random Forest explanations?
c. Why is DeepSHAP still approximate even though it uses the network structure?
d. How might the choice of background proteins affect the DeepSHAP results?

> *Your answer here*


## Section 7: Synthesis across SHAP methods

We now compare the three SHAP analyses:
- **Kernel SHAP** on the Random Forest
- **TreeSHAP** on the Random Forest
- **DeepSHAP** on the MLP

The goal is not to force perfect agreement, but to see which biological signals are **robust across methods** and which vary depending on the model or the approximation.

> ✏️ **STUDENT TASK 7.1:** Run the comparison cell below, then answer the synthesis questions that follow.  
> Pay particular attention to which features appear in the top 10 of *all three* methods — these are your most reliable candidates for biological follow-up.


In [ ]:


# ── 7.1  Compare SHAP rankings across methods ────────────────────────────────
method_importance = {
    "Kernel SHAP (RF)": mean_kernel,
    "TreeSHAP (RF)": mean_tree_all,
    "DeepSHAP (MLP)": mean_deep,
}

corr_table = pd.DataFrame(index=method_importance.keys(), columns=method_importance.keys(), dtype=float)
for row_name, row_values in method_importance.items():
    for col_name, col_values in method_importance.items():
        corr_table.loc[row_name, col_name] = spearmanr(row_values, col_values).statistic

print("Spearman rank correlation of mean |SHAP| feature rankings")
display(corr_table.style.format("{:.3f}"))

top15_ref = np.argsort(mean_tree_all)[::-1][:15]
feat_labels_final = [FEATURE_COLS[i][:24] for i in top15_ref]

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
palette = ["#4C72B0", "#55A868", "#C44E52"]

for ax, (method_name, importance), color in zip(axes, method_importance.items(), palette):
    vals = importance[top15_ref]
    vals_norm = vals / (vals.max() + 1e-9)
    ax.barh(feat_labels_final[::-1], vals_norm[::-1], color=color, edgecolor="black", alpha=0.9)
    ax.set_title(method_name)
    ax.set_xlabel("Normalized importance")

plt.suptitle("Top features across SHAP methods (reference order = TreeSHAP)", y=1.02)
plt.tight_layout()
plt.show()

consensus_top10 = set(FEATURE_COLS[i] for i in np.argsort(mean_kernel)[::-1][:10])
consensus_top10 &= set(FEATURE_COLS[i] for i in np.argsort(mean_tree_all)[::-1][:10])
consensus_top10 &= set(FEATURE_COLS[i] for i in np.argsort(mean_deep)[::-1][:10])
print(f"Features appearing in the top 10 of all three SHAP analyses: {sorted(consensus_top10)}")


**Q7.1 -**
a. Which features appear consistently across Kernel SHAP, TreeSHAP, and DeepSHAP?
b. When two methods disagree, how do you decide whether that difference is meaningful or just approximation noise?
c. Fill in the table below.

| Method | Main strength | Main weakness | Best use case |
|--------|---------------|---------------|---------------|
| Kernel SHAP | | | |
| TreeSHAP | | | |
| DeepSHAP | | | |

> *Your answer here*

> ✏️ **Student activity: beeswarm comparisons**
>
> Create a beeswarm for `shap_exp_kernel_rf`, `shap_exp_tree`, and `shap_exp_deep`.
> For each plot, compare the feature ranking from the beeswarm with the ranking from the bar chart.
> Which features are globally important, and which are simply variable across samples?



---
## Wrap-up

This notebook now uses SHAP in the order that is usually easiest to teach:
1. **One local prediction first** with a force plot
2. **Kernel SHAP sensitivity checks** for background size and `nsamples`
3. **TreeSHAP** as the fast, exact default for tree models
4. **DeepSHAP** as the neural-network counterpart

### Main takeaways

1. **Force plots are local decompositions.** They show how we move from a baseline prediction to one protein's final prediction.
2. **Kernel SHAP is practical but sensitive.** Its results depend on the background set and on `nsamples`, so both should be sanity-checked.
3. **TreeSHAP is usually the best choice for tree ensembles.** It is faster and more stable because it uses the tree structure directly.
4. **DeepSHAP is useful for neural networks, but still approximate.** Its attributions depend on the chosen reference background.
5. **Beeswarms are summaries, not explanations by themselves.** They are most useful after students already understand local SHAP values from force plots or contribution tables.

*End of Assignment 1*
